# Document Ingestion and Vector Indexing Pipeline

This notebook demonstrates the document ingestion lifecycle, from raw data extraction to persistent vector storage. The process is designed to be idempotent and efficient, utilizing content-based hashing to prevent redundant indexing.

## 1. Document Loading

We utilize standardized loaders for PDF and text-based documents. The `load_text_and_pdfs` function recursively scans the data directory and extracts content into LangChain `Document` objects.

In [1]:
import sys
import os
sys.path.append('..') # Add root to path to access src/

from src.config import AppConfig
from src.ingestion import load_text_and_pdfs

cfg = AppConfig()
documents = load_text_and_pdfs(cfg.paths.data_dir)
print(f"Total documents extracted: {len(documents)}")
if documents:
    print(f"Sample Metadata: {documents[0].metadata}")

c:\Users\owner\Desktop\python\RAG\rag-1\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\owner\Desktop\python\RAG\rag-1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total documents extracted: 5
Sample Metadata: {'source': 'C:\\Users\\owner\\Desktop\\python\\RAG\\rag-1\\data\\text_files\\machine_learning.txt'}


## 2. Text Partitioning (Chunking)

To optimize retrieval precision, documents are partitioned into smaller, overlapping segments. This ensures that the context provided to the LLM is focused and relevant.

In [2]:
from src.ingestion import split_documents

chunks = split_documents(documents, chunk_size=500, chunk_overlap=100)
print(f"Total segments generated: {len(chunks)}")

Total segments generated: 19


## 3. Embedding Generation and Persistent Indexing

The system transforms text segments into high-dimensional embeddings and stores them in a persistent vector database. We use content-based hashing to generate stable IDs, ensuring that only modified or new content is updated.

In [3]:
from src.embeddings import EmbeddingManager
from src.vectorstores import ChromaVectorStore

# Initialize embedding model
embed_manager = EmbeddingManager()
embed_manager.load_model()

# Initialize vector store
vector_store = ChromaVectorStore()

# Generate embeddings
texts = [doc.page_content for doc in chunks]
embeddings = embed_manager.generate_embeddings(texts)

# Perform idempotent 'upsert' operation
vector_store.add_documents(chunks, embeddings)
print(f"Final record count in store: {vector_store.count()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2292.60it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Final record count in store: 77
